# Time Tracker — 01: Getting Started

This notebook walks through the **Time Tracker** app from the inside out.  
We populate a demo database with realistic example data, then call the same
functions that each Streamlit page uses — so you can see exactly what is
happening under the hood before opening the browser.

**What you will learn:**
1. How subjects and time entries are stored in SQLite via `TimesheetDB`
2. How **Page 1 (Weekly Table)** pivots raw entries into the editable grid
3. How **Page 2 (Weekly Report)** aggregates and compares data to long-term averages
4. How **Page 3 (Reflection)** stores free-text notes and weekly goals
5. How **Page 4 (Goal Review)** evaluates goals against actual logged hours
6. How to launch the full Streamlit app

> **Requirements:** run `pip install -r ../requirements.txt` before executing this notebook.

In [1]:
!pip install -r ../requirements.txt -q

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from datetime import date, timedelta
import pandas as pd

from tracker.database import TimesheetDB
from tracker.models import Subject, TimeEntry, Reflection, Goal, GoalOutcome
from tracker.analytics import (
    week_monday, week_pivot, fmt_hours,
    aggregate_by_label, aggregate_by_subject,
    weekly_totals_over_range, label_averages_over_range,
    compare_to_averages, narrative_summary, evaluate_goals,
    _DAY_ABBR,
)

DEMO_DB = 'demo.db'

# Start from a clean slate
if os.path.exists(DEMO_DB):
    os.remove(DEMO_DB)

print('Python', sys.version[:6])
print('Demo database:', os.path.abspath(DEMO_DB))

## 2. Subjects — the Building Blocks

Every time entry belongs to a **Subject**.  
Each subject carries two label tiers:

| Field | Purpose | Example |
|---|---|---|
| `name` | Unique identifier shown in the table | `Deep Learning` |
| `low_level_label` | Fine-grained category | `research` |
| `high_level_label` | Broad life-area category | `Work` |

This two-tier structure lets Page 2 aggregate hours both by broad life area
(*Work vs. Health vs. Learning*) and by specific activity.

In [4]:
subjects_data = [
    Subject(name='Deep Learning',     low_level_label='research',    high_level_label='Work'),
    Subject(name='Literature Review', low_level_label='research',    high_level_label='Work'),
    Subject(name='Team Meeting',      low_level_label='admin',       high_level_label='Work'),
    Subject(name='Running',           low_level_label='sport',       high_level_label='Health'),
    Subject(name='Meditation',        low_level_label='mindfulness', high_level_label='Health'),
    Subject(name='Side Projects',     low_level_label='coding',      high_level_label='Learning'),
    Subject(name='Reading',           low_level_label='books',       high_level_label='Learning'),
]

with TimesheetDB(DEMO_DB) as db:
    for s in subjects_data:
        db.add_subject(s)
    saved = db.get_all_subjects()

print(f'Stored {len(saved)} subjects:\n')
print(f"{'ID':<4} {'Name':<20} {'Low Label':<15} {'High Label'}")
print('-' * 55)
for s in saved:
    print(f"{s.id:<4} {s.name:<20} {s.low_level_label:<15} {s.high_level_label}")

Stored 7 subjects:

ID   Name                 Low Label       High Label
-------------------------------------------------------
1    Deep Learning        research        Work
2    Literature Review    research        Work
5    Meditation           mindfulness     Health
7    Reading              books           Learning
4    Running              sport           Health
6    Side Projects        coding          Learning
3    Team Meeting         admin           Work


## 3. Time Entries — Logging Hours

A `TimeEntry` records how many hours were spent on a subject on a given date.  
We insert four weeks of realistic data so the analytics pages have enough
history to show trends and long-term averages.

In [5]:
today = date.today()
this_monday = week_monday(today)

# 4 weeks of example schedules (hours per subject per weekday)
# Format: {subject_name: [Mon, Tue, Wed, Thu, Fri, Sat, Sun]}
weekly_schedules = [
    {  # current week
        'Deep Learning':     [4.0, 3.5, 4.0, 3.0, 2.5, 1.0, 0.0],
        'Literature Review': [1.5, 2.0, 1.0, 2.0, 1.5, 0.0, 0.0],
        'Team Meeting':      [1.0, 0.0, 2.0, 0.0, 1.0, 0.0, 0.0],
        'Running':           [0.0, 1.0, 0.0, 1.0, 0.0, 1.5, 1.0],
        'Meditation':        [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
        'Side Projects':     [0.0, 1.0, 0.0, 1.5, 0.0, 2.0, 3.0],
        'Reading':           [0.5, 0.5, 0.5, 0.5, 0.5, 1.0, 1.5],
    },
    {  # 1 week ago
        'Deep Learning':     [3.5, 4.0, 3.0, 4.0, 3.5, 0.5, 0.0],
        'Literature Review': [2.0, 1.5, 2.0, 1.0, 2.0, 0.0, 0.0],
        'Team Meeting':      [0.0, 1.0, 0.0, 2.0, 0.0, 0.0, 0.0],
        'Running':           [1.0, 0.0, 1.0, 0.0, 1.0, 2.0, 1.5],
        'Meditation':        [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
        'Side Projects':     [1.0, 0.0, 1.5, 0.0, 1.0, 3.0, 2.0],
        'Reading':           [0.5, 1.0, 0.5, 0.5, 1.0, 1.5, 1.0],
    },
    {  # 2 weeks ago
        'Deep Learning':     [4.5, 4.0, 3.5, 3.0, 4.0, 1.0, 0.0],
        'Literature Review': [1.0, 2.0, 1.5, 2.0, 1.0, 0.0, 0.0],
        'Team Meeting':      [1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0],
        'Running':           [0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.5],
        'Meditation':        [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
        'Side Projects':     [0.0, 0.5, 1.0, 0.0, 1.5, 2.5, 3.5],
        'Reading':           [1.0, 0.5, 0.5, 1.0, 0.5, 1.0, 1.5],
    },
    {  # 3 weeks ago
        'Deep Learning':     [3.0, 3.5, 4.0, 3.5, 3.0, 0.0, 0.0],
        'Literature Review': [2.0, 1.5, 1.0, 1.5, 2.0, 0.0, 0.0],
        'Team Meeting':      [1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0],
        'Running':           [1.0, 0.0, 1.0, 0.0, 1.0, 1.5, 1.0],
        'Meditation':        [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
        'Side Projects':     [0.5, 0.0, 1.0, 1.0, 0.0, 2.0, 2.5],
        'Reading':           [0.5, 0.5, 1.0, 0.5, 0.5, 1.5, 1.0],
    },
]

with TimesheetDB(DEMO_DB) as db:
    all_subjects = {s.name: s for s in db.get_all_subjects()}
    total_entries = 0
    for week_offset, schedule in enumerate(weekly_schedules):
        monday = this_monday - timedelta(weeks=week_offset)
        for subj_name, hours_list in schedule.items():
            subj = all_subjects[subj_name]
            for day_idx, hours in enumerate(hours_list):
                if hours > 0:
                    db.add_entry(TimeEntry(
                        date=monday + timedelta(days=day_idx),
                        subject_id=subj.id,
                        duration_hours=hours,
                    ))
                    total_entries += 1

print(f'Inserted {total_entries} time entries across 4 weeks')
print(f'Date range: {this_monday - timedelta(weeks=3)}  →  {this_monday + timedelta(days=6)}')

Inserted 148 time entries across 4 weeks
Date range: 2026-03-23  →  2026-04-19


## 4. Page 1 — Weekly Table

Page 1 shows an editable pivot grid: one row per subject, one column per weekday.
The underlying function is `week_pivot()`, which:
1. Fetches all entries for the selected week
2. Pivots them into a `(subject × day)` matrix
3. Adds a `Total` column (row sum)

You can edit any day cell directly in the browser — the app writes the change
back to SQLite immediately.

In [6]:
with TimesheetDB(DEMO_DB) as db:
    entries_this_week = db.get_entries_for_week(this_monday)

pivot = week_pivot(entries_this_week, this_monday)

print(f'Week of {this_monday}  →  {this_monday + timedelta(days=6)}')
print(f'{len(entries_this_week)} raw entries  →  {len(pivot)} subject rows\n')
print(pivot.to_string(index=False))

Week of 2026-04-13  →  2026-04-19
36 raw entries  →  7 subject rows

          Subject   Low Label High Label  Mon  Tue  Wed  Thu  Fri  Sat  Sun  Total
    Deep Learning    research       Work  4.0  3.5  4.0  3.0  2.5  1.0  0.0   18.0
Literature Review    research       Work  1.5  2.0  1.0  2.0  1.5  0.0  0.0    8.0
    Side Projects      coding   Learning  0.0  1.0  0.0  1.5  0.0  2.0  3.0    7.5
          Reading       books   Learning  0.5  0.5  0.5  0.5  0.5  1.0  1.5    5.0
          Running       sport     Health  0.0  1.0  0.0  1.0  0.0  1.5  1.0    4.5
     Team Meeting       admin       Work  1.0  0.0  2.0  0.0  1.0  0.0  0.0    4.0
       Meditation mindfulness     Health  0.5  0.5  0.5  0.5  0.5  0.5  0.5    3.5


In [7]:
# Daily totals (the purple DAILY TOTAL row at the bottom of the table)
day_cols = [d for d in _DAY_ABBR if d in pivot.columns]
daily_totals = pivot[day_cols].sum()
week_total = daily_totals.sum()

print('Daily totals:')
for day, total in daily_totals.items():
    bar = '█' * int(total)
    print(f'  {day}  {total:5.2f}h  {bar}')
print(f'  {"─"*25}')
print(f'  Week  {week_total:5.2f}h  →  {fmt_hours(week_total)}')

Daily totals:
  Mon   7.50h  ███████
  Tue   8.50h  ████████
  Wed   8.00h  ████████
  Thu   8.50h  ████████
  Fri   6.00h  ██████
  Sat   6.00h  ██████
  Sun   6.00h  ██████
  ─────────────────────────
  Week  50.50h  →  50h 30m


## 5. Page 2 — Weekly Report

Page 2 is the analytics dashboard. It answers three questions:

1. **How did I spend my time this week?** — by high-level and low-level label
2. **How does this week compare to my long-term average?** — delta per category
3. **How is my total tracked time trending?** — week-over-week chart

It also generates a short plain-text **narrative summary**.

In [8]:
# --- 5a. Hours by high-level label (life area) ---
high_agg = aggregate_by_label(entries_this_week, level='high')

print('This week — hours by life area:')
print(f"  {'Category':<12} {'Hours':>6}  Bar")
print('  ' + '─' * 35)
for _, row in high_agg.iterrows():
    bar = '█' * int(row['total_hours'])
    print(f"  {row['label']:<12} {row['total_hours']:>6.2f}h  {bar}")

This week — hours by life area:
  Category      Hours  Bar
  ───────────────────────────────────
  Work          30.00h  ██████████████████████████████
  Learning      12.50h  ████████████
  Health         8.00h  ████████


In [9]:
# --- 5b. Hours by low-level label (specific activity) ---
low_agg = aggregate_by_label(entries_this_week, level='low')

print('This week — hours by specific activity:')
print(f"  {'Activity':<15} {'Hours':>6}")
print('  ' + '─' * 25)
for _, row in low_agg.iterrows():
    print(f"  {row['label']:<15} {row['total_hours']:>6.2f}h")

This week — hours by specific activity:
  Activity         Hours
  ─────────────────────────
  research         26.00h
  coding            7.50h
  books             5.00h
  sport             4.50h
  admin             4.00h
  mindfulness       3.50h


In [10]:
# --- 5c. Long-term averages (across all 4 weeks in the demo DB) ---
from_date = this_monday - timedelta(weeks=3)
to_date   = this_monday + timedelta(days=7)

with TimesheetDB(DEMO_DB) as db:
    all_entries = db.get_entries_for_range(from_date, to_date)

avg_df = label_averages_over_range(all_entries, level='high')

print(f'Long-term averages across {avg_df["n_weeks"].iloc[0]:.0f} weeks:')
print(f"  {'Category':<12} {'Avg h/week':>10}")
print('  ' + '─' * 25)
for _, row in avg_df.iterrows():
    print(f"  {row['label']:<12} {row['avg_hours_per_week']:>10.2f}h")

Long-term averages across 4 weeks:
  Category     Avg h/week
  ─────────────────────────
  Work              29.88h
  Learning          13.62h
  Health             8.75h


In [11]:
# --- 5d. This week vs. long-term average ---
comparison = compare_to_averages(high_agg, avg_df)

print('This week vs. long-term average:')
print(f"  {'Category':<12} {'This wk':>8} {'Avg':>8} {'Diff':>8} {'%':>7}")
print('  ' + '─' * 48)
for _, row in comparison.iterrows():
    pct = f"{row['pct_diff']:+.0f}%" if pd.notna(row['pct_diff']) else '   n/a'
    arrow = '▲' if row['diff_hours'] > 0 else ('▼' if row['diff_hours'] < 0 else '=')
    print(f"  {row['label']:<12} {row['total_hours']:>7.2f}h {row['avg_hours_per_week']:>7.2f}h {row['diff_hours']:>+7.2f}h {pct:>6}  {arrow}")

This week vs. long-term average:
  Category      This wk      Avg     Diff       %
  ────────────────────────────────────────────────
  Work           30.00h   29.88h   +0.12h    +0%  ▲
  Learning       12.50h   13.62h   -1.12h    -8%  ▼
  Health          8.00h    8.75h   -0.75h    -9%  ▼


In [12]:
# --- 5e. Week-over-week totals ---
weekly_trend = weekly_totals_over_range(all_entries)

print('Week-over-week tracked hours:')
for _, row in weekly_trend.iterrows():
    bar = '█' * int(row['total_hours'] / 2)
    print(f"  {str(row['week_start'])}  {row['total_hours']:5.2f}h  {bar}")

Week-over-week tracked hours:
  2026-03-23  50.50h  █████████████████████████
  2026-03-30  53.50h  ██████████████████████████
  2026-04-06  54.50h  ███████████████████████████
  2026-04-13  50.50h  █████████████████████████


In [13]:
# --- 5f. Narrative summary (the auto-generated text shown at the top of Page 2) ---
summary = narrative_summary(entries_this_week, avg_df=avg_df, level='high')
print('Auto-generated narrative:\n')
print(' ', summary)

Auto-generated narrative:

  You logged 50h 30m of tracked time this week across 3 categories.  Work received the most attention (30.0h).  Health received the least attention (8.0h).  Below long-term average: Learning, Health.


## 6. Page 3 — Reflection

Page 3 lets you write a free-text weekly reflection with three fields:
- **Strengths** — what went well this week
- **Weaknesses** — what to improve
- **Next-week plan** — concrete intentions

You can also set **goals** for next week, optionally linked to a subject
with a target number of hours (used by Page 4 for auto-evaluation).

In [14]:
# Save a reflection for this week
reflection = Reflection(
    week_start=this_monday,
    strengths='Maintained consistent deep learning sessions every morning. '
              'Finished two papers in the literature review.',
    weaknesses='Team meetings ran over — need to set stricter time limits. '
               'Side project hours lower than planned on weekdays.',
    next_week_plan='Block 09:00–12:00 daily for deep learning. '
                   'Add one extra running session on Wednesday.',
)

with TimesheetDB(DEMO_DB) as db:
    db.upsert_reflection(reflection)
    saved_ref = db.get_reflection(this_monday)

print(f'Reflection for week of {saved_ref.week_start}\n')
print(f'Strengths:       {saved_ref.strengths[:80]}...')
print(f'Weaknesses:      {saved_ref.weaknesses[:80]}...')
print(f'Next-week plan:  {saved_ref.next_week_plan[:80]}...')

Reflection for week of 2026-04-13

Strengths:       Maintained consistent deep learning sessions every morning. Finished two papers ...
Weaknesses:      Team meetings ran over — need to set stricter time limits. Side project hours lo...
Next-week plan:  Block 09:00–12:00 daily for deep learning. Add one extra running session on Wedn...


In [15]:
# Set goals for next week
next_monday = this_monday + timedelta(weeks=1)

with TimesheetDB(DEMO_DB) as db:
    all_subjects = {s.name: s for s in db.get_all_subjects()}

goals_data = [
    Goal(
        week_start=next_monday,
        description='Log at least 20h of deep learning work',
        target_hours=20.0,
        subject_id=all_subjects['Deep Learning'].id,
    ),
    Goal(
        week_start=next_monday,
        description='Run 3 times this week',
        target_hours=3.0,
        subject_id=all_subjects['Running'].id,
    ),
    Goal(
        week_start=next_monday,
        description='Read every day before bed',
        target_hours=None,   # no specific hour target
        subject_id=all_subjects['Reading'].id,
    ),
]

with TimesheetDB(DEMO_DB) as db:
    for g in goals_data:
        db.add_goal(g)
    saved_goals = db.get_goals_for_week(next_monday)

print(f'{len(saved_goals)} goals set for week of {next_monday}:\n')
for g in saved_goals:
    target = f'{g.target_hours}h target' if g.target_hours else 'no hour target'
    print(f'  [{g.id}] {g.description}')
    print(f'       subject: {g.subject_name}  |  {target}')

3 goals set for week of 2026-04-20:

  [1] Log at least 20h of deep learning work
       subject: Deep Learning  |  20.0h target
  [2] Run 3 times this week
       subject: Running  |  3.0h target
  [3] Read every day before bed
       subject: Reading  |  no hour target


## 7. Page 4 — Goal Review

At the end of each week, Page 4 shows the goals you set the previous week
and lets you evaluate them:
- **Met / Partial / Not met** — manual assessment
- **Auto-evaluated** — if the goal is linked to a subject with a target, the app
  computes actual hours automatically and compares them to the target

Here we simulate a completed next week and evaluate the goals.

In [16]:
# Simulate logged hours for next week (so we can evaluate the goals)
next_week_hours = {
    'Deep Learning': [4.5, 4.0, 4.0, 3.5, 4.0, 0.0, 0.0],   # 20h total — goal met!
    'Running':       [0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0],   # 3 sessions — goal met!
    'Reading':       [0.5, 0.5, 0.0, 0.5, 0.5, 1.0, 0.5],   # daily — partial
}

with TimesheetDB(DEMO_DB) as db:
    all_subjects = {s.name: s for s in db.get_all_subjects()}
    for subj_name, hours_list in next_week_hours.items():
        subj = all_subjects[subj_name]
        for day_idx, hours in enumerate(hours_list):
            if hours > 0:
                db.add_entry(TimeEntry(
                    date=next_monday + timedelta(days=day_idx),
                    subject_id=subj.id,
                    duration_hours=hours,
                ))

    next_entries = db.get_entries_for_week(next_monday)

print(f'Logged {len(next_entries)} entries for next week')

Logged 14 entries for next week


In [17]:
# Evaluate goals against actual hours
with TimesheetDB(DEMO_DB) as db:
    goals = db.get_goals_for_week(next_monday)

evaluated = evaluate_goals(goals, next_entries)

MET_LABEL = {0: 'Not met', 1: 'Met', 2: 'Partial'}

print('Goal evaluation results:\n')
for item in evaluated:
    g = item['goal']
    actual = item['actual_hours']
    if g.target_hours and actual is not None:
        met = 1 if actual >= g.target_hours else (2 if actual >= g.target_hours * 0.7 else 0)
        status = MET_LABEL[met]
        print(f'  Goal : {g.description}')
        print(f'  Target: {g.target_hours}h  |  Actual: {actual:.1f}h  |  Status: {status}')
    else:
        actual_str = f'{actual:.1f}h' if actual else 'not tracked'
        print(f'  Goal : {g.description}')
        print(f'  Actual: {actual_str}  |  Status: manual review needed')
    print()

Goal evaluation results:

  Goal : Log at least 20h of deep learning work
  Target: 20.0h  |  Actual: 20.0h  |  Status: Met

  Goal : Run 3 times this week
  Target: 3.0h  |  Actual: 3.0h  |  Status: Met

  Goal : Read every day before bed
  Actual: 3.5h  |  Status: manual review needed



In [18]:
# Save outcomes back to the DB (what Page 4 does when you click Save)
with TimesheetDB(DEMO_DB) as db:
    goals = db.get_goals_for_week(next_monday)
    next_entries = db.get_entries_for_week(next_monday)
    evaluated = evaluate_goals(goals, next_entries)

    for item in evaluated:
        g = item['goal']
        actual = item['actual_hours']
        if g.target_hours and actual is not None:
            met = 1 if actual >= g.target_hours else (2 if actual >= g.target_hours * 0.7 else 0)
        else:
            met = 2
        db.upsert_goal_outcome(GoalOutcome(
            goal_id=g.id,
            actual_hours=actual,
            met=met,
            notes='Auto-evaluated by notebook demo',
        ))

print('Goal outcomes saved to DB.')

Goal outcomes saved to DB.


## 8. Database Summary

A final look at everything stored in `demo.db`.

In [19]:
import sqlite3

with sqlite3.connect(DEMO_DB) as conn:
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    print('Tables in demo.db:\n')
    for (tbl,) in tables:
        count = conn.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
        print(f'  {tbl:<20} {count:>4} rows')

Tables in demo.db:

  subjects                7 rows
  sqlite_sequence         5 rows
  time_entries          162 rows
  reflections             1 rows
  goals                   3 rows
  goal_outcomes           3 rows


## 9. Running the Full App

Everything shown above runs automatically in the browser when you start the app.
The Streamlit pages are thin wrappers around the same `tracker.*` functions used here.

```bash
# From the time_tracker/ directory:
streamlit run app.py
```

**On Windows:** double-click `launch.vbs` (no terminal needed).

The app opens at `http://localhost:8501`. Data is stored locally in `data/timesheet.db` — no cloud, no account.

| Page | What it does | Key function |
|---|---|---|
| 1 · Weekly Table | Editable pivot grid — log hours, add/delete subjects | `week_pivot()` |
| 2 · Weekly Report | Analytics dashboard — totals, trends, averages | `aggregate_by_label()`, `compare_to_averages()` |
| 3 · Reflection | Free-text notes + goal setting for next week | `upsert_reflection()`, `add_goal()` |
| 4 · Goal Review | Evaluate last week's goals against actual hours | `evaluate_goals()`, `upsert_goal_outcome()` |

## Summary

In this notebook we:
- Created 7 subjects across 3 life areas (Work, Health, Learning)
- Inserted 4 weeks of realistic time entries into a local SQLite database
- Reproduced the **Weekly Table** pivot (Page 1) and computed daily totals
- Reproduced the **Weekly Report** aggregations, trend, and narrative summary (Page 2)
- Saved a **Reflection** with strengths, weaknesses, and a next-week plan (Page 3)
- Set 3 goals, simulated the following week, and auto-evaluated outcomes (Page 4)

The demo database `demo.db` in this folder can be deleted at any time — it has no effect on the main app database at `data/timesheet.db`.